# Visual-token budget sweep — DeepSeek-OCR-2 on TexOCR-eval pages

Three native configurations, whole papers, resume-safe.

| run name | `base_size` | `image_size` | `crop_mode` | visual tokens |
|---|---|---|---|---|
| `Base` | 1024 | 1024 | False | 256 |
| `Large` | 1280 | 1280 | False | 400 |
| `Gundam` | 1024 | 768 | True | 256 + n x 144, n = 2..6 (aspect-driven) |

`infer()` in `modeling_deepseekocr2.py` (verified on the HF revision in use):

```python
def infer(self, tokenizer, prompt='', image_file='', output_path='', base_size=1024,
          image_size=640, crop_mode=True, test_compress=False, save_results=False,
          eval_mode=False):
    os.makedirs(output_path, exist_ok=True)   # unconditionally -> '' raises
```

So every call here passes `output_path="/tmp/ds_out"` (a non-empty dir), `eval_mode=True`
(otherwise it streams tokens and returns `None`, and `runs.csv` logs empty predictions),
and `save_results=False`. Two defaults worth pinning down:

- `image_size` defaults to **640**, not 768 — omitting it silently runs 100-token crops
  instead of 144, so it is passed explicitly on every call.
- the `eval_mode` branch generates with `no_repeat_ngram_size=35`, the streaming branch
  with `20`. Only `eval_mode=True` returns text (the others fall through to `None`), so
  these numbers are close to, not bit-identical with, the repo's interactive demo — say
  so in the figure caption.


Outputs (on Drive, so a timeout costs nothing):

- `runs.csv` — one row per (mode, page), appended and fsynced immediately; rerunning skips
  pairs already present, so a 4–6 h sweep can span several sessions.
- `outputs/<mode>/<page_id>.tex` — raw model output, one file per page.

Compilation is **not** done here: run `scripts/score.py --group-by paper` on a CPU runtime
afterwards (pdflatex on a GPU runtime burns units for nothing).

In [ ]:
# 1. Environment. Runs once per session (~4 min).
#
# Ignore the pip dependency-conflict report about diffusers/gradio needing a newer
# huggingface-hub. transformers==4.46.3 pins hub<1.0 by design and nothing here imports
# those two packages. If Colab offers "Restart runtime", take it: installs persist, then
# continue from cell 2.
!pip install -q "transformers==4.46.3" "tokenizers<0.21" "huggingface-hub<1.0" accelerate addict easydict rapidfuzz pandas matplotlib
%env TOKENIZERS_PARALLELISM=false
import torch, transformers, tokenizers, huggingface_hub
print("torch", torch.__version__, "| cuda", torch.version.cuda, "| gpu", torch.cuda.get_device_name(0))
print("vram GB", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
print("transformers", transformers.__version__, "| tokenizers", tokenizers.__version__,
      "| hub", huggingface_hub.__version__)
assert transformers.__version__.startswith("4.46"), "another transformers got installed"
assert torch.cuda.is_available(), "GPU runtime not selected (Runtime -> Change runtime type -> L4)"

In [ ]:
# 2. Paths. Everything durable lives on Drive; code is a fresh clone.
from google.colab import drive
drive.mount("/content/drive")

import os, json, pathlib, subprocess
WORK = pathlib.Path("/content/drive/MyDrive/ocr-layout-budget")   # <-- edit if different
REPO = pathlib.Path("/content/ocr-layout-token-budget")
DATA = WORK / "data"
OUT  = WORK / "outputs"
RUNS = WORK / "runs.csv"

# one-shot unpack of the uploaded archive (no-op once data/ exists).
# --warning=no-unknown-keyword silences macOS pax headers (LIBARCHIVE.xattr.*),
# which GNU tar ignores anyway; output is printed only on failure.
if not (DATA / "pages.jsonl").exists():
    r = subprocess.run(["tar", "--warning=no-unknown-keyword", "-xzf",
                        str(WORK / "data.tar.gz"), "-C", str(WORK)],
                       capture_output=True, text=True)
    print("untar rc", r.returncode, (r.stderr or "").strip()[-400:])
    assert (DATA / "pages.jsonl").exists(), "untar produced no pages.jsonl -- re-upload data.tar.gz?"

for p in (OUT / "Base", OUT / "Large", OUT / "Gundam", WORK / "figs"):
    p.mkdir(parents=True, exist_ok=True)
if not REPO.exists():
    print(subprocess.run(["git", "clone", "-q",
                          "https://github.com/hluaguo/ocr-layout-token-budget", str(REPO)],
                         capture_output=True, text=True).stderr)

pages = [json.loads(l) for l in open(DATA / "pages.jsonl")]

# Scope knob. The manifest holds 30 papers / 236 pages; paper-major trimming keeps
# every selected paper complete, so paper-level NED stays exact. Lower it when the
# measured generation speed does not fit the session budget (see the ETA cell).
PAPERS_PER_BUCKET = 10
def trim_per_bucket(rows, n):
    keep = {}
    for r in sorted(rows, key=lambda x: (x["bucket"], x["paper_id"], x["page"])):
        s = keep.setdefault(r["bucket"], set())
        if r["paper_id"] in s or len(s) < n:
            s.add(r["paper_id"])
    kept = {pid for s in keep.values() for pid in s}
    return [r for r in rows if r["paper_id"] in kept]
pages = trim_per_bucket(pages, PAPERS_PER_BUCKET)
import collections
print("scoped to", len(pages), "pages /", len({p['paper_id'] for p in pages}), "papers:",
      dict(collections.Counter(p['bucket'] for p in pages)))
print(len(pages), "pages,", len({p['paper_id'] for p in pages}), "papers")
missing = [p['page_id'] for p in pages if not (DATA / p['image']).exists()]
print("missing images:", len(missing), missing[:3])

In [ ]:
# 3. Load the model (once per session, ~2 min).
import inspect, sys
from transformers import AutoModel, AutoTokenizer

MODEL_ID = "deepseek-ai/DeepSeek-OCR-2"
tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModel.from_pretrained(
    MODEL_ID, trust_remote_code=True, use_safetensors=True,
    _attn_implementation="eager",   # flash_attention_2 if the wheel is present
).eval().cuda().to(torch.bfloat16)

print("infer signature:", inspect.signature(model.infer))
PROMPT = "<image>\n<|grounding|>Convert the document to latex."
os.makedirs("/tmp/ds_out", exist_ok=True)   # infer() makedirs() this unconditionally

### Exact crop counts from the model's own tiling

`modeling_deepseekocr2.dynamic_preprocess(image, min_num=2, max_num=6, image_size=768)`
decides the crop grid from the page's aspect ratio, so importing it gives the exact block
count instead of an upper bound: `tokens = (base_size/64)^2 + blocks * (image_size/64)^2`.

A forward hook on the vision tower runs as an independent cross-check; the two must agree,
and both are written to `runs.csv` (`hook_crops`, `token_source`).

In [ ]:
CONFIGS = {
    "Base":   dict(base_size=1024, image_size=1024, crop_mode=False),
    "Large":  dict(base_size=1280, image_size=1280, crop_mode=False),
    "Gundam": dict(base_size=1024, image_size=768,  crop_mode=True),
}
PATCH = 64            # a 64 px patch -> one visual token
N_CROP_MIN, N_CROP_MAX = 2, 6

mod = sys.modules[model.__class__.__module__]
dynamic_preprocess = getattr(mod, "dynamic_preprocess", None)
print("dynamic_preprocess:", "imported" if dynamic_preprocess else "NOT FOUND (fallback)")

from PIL import Image
def crop_count(image_path, cfg):
    if not cfg["crop_mode"]:
        return 0, "no_crop"
    if dynamic_preprocess is None:
        return N_CROP_MAX, "assumed_max"
    im = Image.open(image_path).convert("RGB")
    blocks = dynamic_preprocess(im, min_num=N_CROP_MIN, max_num=N_CROP_MAX,
                               image_size=cfg["image_size"])
    return len(blocks), "dynamic_preprocess"

def visual_tokens(cfg, blocks):
    base = (cfg["base_size"] // PATCH) ** 2
    return base if not cfg["crop_mode"] else base + blocks * (cfg["image_size"] // PATCH) ** 2

for name, cfg in CONFIGS.items():
    lo = visual_tokens(cfg, 0 if not cfg["crop_mode"] else N_CROP_MIN)
    hi = visual_tokens(cfg, 0 if not cfg["crop_mode"] else N_CROP_MAX)
    print(f"{name:7s} {cfg}  ->  {lo}..{hi} tokens")

# independent cross-check: how many crop tensors reach the vision tower
VISION_ATTRS = ("vision_model", "vision_tower", "vision_encoder", "deepencoder", "encoder")
HOOK_STATE = {"n_crops": None}
def _hook_factory():
    def hook(module, args, kwargs, output):
        for obj in list(args) + list(kwargs.values()):
            if torch.is_tensor(obj) and obj.dim() == 5:      # (B, n, C, H, W)
                HOOK_STATE["n_crops"] = int(obj.shape[1]); return
            if isinstance(obj, (list, tuple)) and len(obj) and torch.is_tensor(obj[0]) and obj[0].dim() == 4:
                HOOK_STATE["n_crops"] = len(obj); return
    return hook

hooked = []
for attr in VISION_ATTRS:
    m = getattr(model, attr, None)
    if m is not None and hasattr(m, "register_forward_hook"):
        m.register_forward_hook(_hook_factory()); hooked.append(attr)
print("hooks on:", hooked or "none (dynamic_preprocess is the primary source)")

In [ ]:
# 4. Resume-safe index of what already ran.
import csv, os
FIELDS = ["mode", "page_id", "paper_id", "page", "bucket", "base_size", "image_size",
          "crop_mode", "num_crops", "hook_crops", "measured_visual_tokens", "token_source",
          "pred_path", "gold_paper", "gen_seconds", "pred_chars"]

def load_done(path):
    done = set()
    if not path.exists():
        return done
    with open(path, newline="", encoding="utf-8", errors="replace") as fh:
        for row in csv.DictReader(fh):
            if row.get("mode") and row.get("page_id") and (row.get("pred_chars") or "0") != "0":
                done.add((row["mode"], row["page_id"]))
    return done

done = load_done(RUNS)
todo = [(m, p) for m in CONFIGS for p in pages if (m, p["page_id"]) not in done]
print("already done:", len(done), "| to run:", len(todo), "| total:", len(CONFIGS) * len(pages))

if not RUNS.exists():
    with open(RUNS, "w", newline="", encoding="utf-8") as fh:
        csv.DictWriter(fh, fieldnames=FIELDS).writeheader()
        fh.flush(); os.fsync(fh.fileno())

In [ ]:
# 5. SWEEP. Each row is flushed + fsynced before the next page, so a Colab timeout
#    loses at most one generation. Re-run this cell to resume.
#
#    SMOKE = True  -> exactly one page across all three modes (3 generations).
#                     Check the ETA cell before flipping it to False.
#    SMOKE = False -> paper-major sweep: for each paper, all pages of one mode, so a
#                     timeout leaves complete papers (paper-level NED stays usable).
import time, traceback

SMOKE = True
RUN_LIMIT = 3 if SMOKE else 120

def sweep_order(rows):
    by_paper = {}
    for r in rows:
        by_paper.setdefault(r["paper_id"], []).append(r)
    order = []
    for pid in sorted(by_paper):
        for m in CONFIGS:
            for r in sorted(by_paper[pid], key=lambda x: x["page"]):
                order.append((m, r))
    return order

target = [(m, pages[0]) for m in CONFIGS] if SMOKE else sweep_order(pages)
target = [x for x in target if (x[0], x[1]["page_id"]) not in done]
print("smoke" if SMOKE else "full sweep", "| queued:", len(target), "of", len(CONFIGS) * len(pages))
ran = 0

for mode, page in todo:
    if ran >= RUN_LIMIT:
        print("run limit reached -- raise RUN_LIMIT or re-run this cell to continue")
        break
    cfg = CONFIGS[mode]
    img = DATA / page["image"]
    dest = OUT / mode / (page["page_id"] + ".tex")
    t0 = time.time()
    HOOK_STATE["n_crops"] = None
    try:
        text = model.infer(
            tok,
            prompt=PROMPT,
            image_file=str(img),
            output_path="/tmp/ds_out",   # infer() makedirs() this unconditionally
            base_size=cfg["base_size"],  # explicit: default image_size is 640, not 768
            image_size=cfg["image_size"],
            crop_mode=cfg["crop_mode"],
            test_compress=False,
            save_results=False,
            eval_mode=True,              # else it streams and returns None
        )
        if isinstance(text, (list, tuple)):
            text = text[0] if text else ""
        text = text or ""
    except Exception:
        print("FAIL", mode, page["page_id"])
        traceback.print_exc(limit=3)
        continue

    if not text.strip():
        print("EMPTY output for", mode, page["page_id"], "-- not logging, check eval_mode")
        continue

    dest.write_text(text, encoding="utf-8")
    blocks, source = crop_count(img, cfg)
    hook_crops = HOOK_STATE["n_crops"]
    if hook_crops is not None and cfg["crop_mode"] and hook_crops != blocks:
        print(f"  ! crop count mismatch: dynamic_preprocess={blocks} hook={hook_crops} ({page['page_id']})")
    tokens = visual_tokens(cfg, blocks)

    with open(RUNS, "a", newline="", encoding="utf-8") as fh:
        csv.DictWriter(fh, fieldnames=FIELDS).writerow({
            "mode": mode, "page_id": page["page_id"], "paper_id": page["paper_id"],
            "page": page["page"], "bucket": page["bucket"],
            "base_size": cfg["base_size"], "image_size": cfg["image_size"],
            "crop_mode": cfg["crop_mode"], "num_crops": blocks, "hook_crops": hook_crops,
            "measured_visual_tokens": tokens, "token_source": source,
            "pred_path": str(dest), "gold_paper": str(DATA / page["paper_gold"]),
            "gen_seconds": round(time.time() - t0, 2), "pred_chars": len(text),
        })
        fh.flush(); os.fsync(fh.fileno())
    ran += 1
    print(f"{ran:3d} {mode:7s} {page['page_id']} {tokens:5d}tok n={blocks} "
          f"{time.time()-t0:5.1f}s {len(text):6d}ch")

print("this cell:", ran, "| total logged:", len(load_done(RUNS)))

In [ ]:
# 6. Smoke gate: run after the 3-page smoke test, before raising RUN_LIMIT.
import pandas as pd
df = pd.read_csv(RUNS)
cols = ["mode", "page_id", "image_size", "crop_mode", "num_crops", "hook_crops",
        "measured_visual_tokens", "token_source", "gen_seconds", "pred_chars"]
print(df[[c for c in cols if c in df.columns]].tail(6).to_string(index=False))
assert (df.pred_chars > 0).all(), "empty predictions: eval_mode/output_path regression"
assert df.measured_visual_tokens.between(256, 1120).all(), "token count outside the expected range"
print("\nOK -- per mode tokens:",
      df.groupby("mode").measured_visual_tokens.agg(["min", "max"]).to_dict("index"))

In [ ]:
# 7. ETA before committing units. Reads gen_seconds from what has run so far.
import pandas as pd, numpy as np
df = pd.read_csv(RUNS)
g = df.groupby("mode").gen_seconds.agg(["count", "median", "mean", "max"]).round(1)
print(g, "\n")

per_page = float(df.gen_seconds.median())
remaining = len(CONFIGS) * len(pages) - len(load_done(RUNS))
hours = per_page * remaining / 3600
print(f"median {per_page:.0f} s/page | remaining {remaining} | projected {hours:.1f} h "
      f"(~{hours * 1.8:.0f} CU on L4 at 1.8 CU/h, + ~0.5 CU/session overhead)")
print("\nrule of thumb: <=2 h -> run as is; 2-5 h -> two sessions; >5 h -> lower"
      " PAPERS_PER_BUCKET in cell 2 and re-run cells 2, 4, 5)")

# truncation check: a page that ran to the 8192-token cap inflates pred_chars and
# biases NED; look for outliers rather than trusting the mean
print("\nlongest outputs:")
print(df.nlargest(3, "pred_chars")[["mode", "page_id", "pred_chars", "gen_seconds"]].to_string(index=False))

In [ ]:
# 8. Deterministic output processing: TexOCR's harness strips code fences; matching it
#    keeps NED comparable. Applied on the way into scoring, not to the raw files.
def process_rsp(r):
    if not r:
        return ""
    for fence in ("```latex", "```bibtex", "```"):
        if r.startswith(fence):
            return r[len(fence):].rstrip("`").rstrip()
    return r

changed = 0
for f in sorted(OUT.rglob("*.tex")):
    raw = f.read_text(encoding="utf-8", errors="replace")
    clean = process_rsp(raw)
    if clean != raw:
        f.write_text(clean, encoding="utf-8"); changed += 1
print("fence-stripped files:", changed)

df = pd.read_csv(RUNS)
print(df.groupby(["bucket", "mode"])[["measured_visual_tokens", "gen_seconds", "pred_chars"]]
        .agg(["count", "mean"]).round(1))
print("\nrows with zero output:", int((df.pred_chars == 0).sum()))

In [ ]:
# 9. Score the completed subset (no pdflatex here; run the compile pass on CPU).
#    --group-by paper concatenates each paper's pages against paper_gold, exactly
#    like TexOCR's harness, so NED is comparable to their edit_distance column.
import subprocess
cmd = ["python", "scripts/score.py", "--runs", str(RUNS), "--pages", str(DATA / "pages.jsonl"),
       "--group-by", "paper", "--no-compile", "--out", str(WORK / "scores.csv"), "--jobs", "8"]
res = subprocess.run(cmd, cwd=REPO, text=True, capture_output=True)
print(res.stderr[-2500:])

import pandas as pd
display(pd.read_csv(WORK / "scores_summary.csv"))
display(pd.read_csv(WORK / "scores_paper_summary.csv"))

In [ ]:
# 10. The plot: measured visual tokens on x, text + layout metrics on y.
#    Compile pass rate appears once the CPU pdflatex pass has run.
import matplotlib.pyplot as plt, pandas as pd

paper = pd.read_csv(WORK / "scores_paper.csv")
fig, ax = plt.subplots(1, 2, figsize=(11, 4), sharex=True)

g = paper.groupby("mode").agg(tokens=("vision_tokens_mean", "mean"), ned=("ned", "mean"))
ax[0].plot(g.tokens, 1 - g.ned, "o-", label="1 - NED (text)")
ax[0].set_ylabel("text fidelity"); ax[0].set_title("Text survives")

if paper["compile_ok"].notna().any():
    c = paper.groupby("mode").agg(tokens=("vision_tokens_mean", "mean"), rate=("compile_ok", "mean"))
    ax[1].plot(c.tokens, c.rate, "s-", label="pdflatex pass rate")
    ax[1].set_title("Structure does not")
else:
    ax[1].text(0.5, 0.5, "run the CPU pdflatex pass", ha="center", va="center")
ax[1].set_ylabel("compile pass rate")
for a in ax:
    a.set_xlabel("measured visual tokens / page"); a.grid(alpha=.3); a.legend()
fig.tight_layout(); fig.savefig(WORK / "figs" / "budget_vs_layout.png", dpi=200)
print("wrote", WORK / "figs" / "budget_vs_layout.png")